<a href="https://colab.research.google.com/github/Harsh-Prajapati54/LLMs---Playbook/blob/main/Document_Loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Document Loader

 in this note book i am going to implement pipline for injection and preprocessing of `pdf` using `PyMupdf `

***Script that ingests a PDF, extracts text, splits into question-level chunks, and saves as structured JSON — zero libraries except PyMuPDF.***

 ### setup

In [ ]:
%%capture
!pip install pymupdf

In [ ]:
import pymupdf
print(pymupdf.__doc__)

PyMuPDF 1.27.2.2: Python bindings for the MuPDF 1.27.2 library.
Python 3.12 running on linux (64-bit).



In [ ]:
%%capture
!pip install -U langchain-text-splitters
!pip install langchain-community

## scripts for opening and preprocessing the pdf document

In [ ]:
# loads the document
doc = pymupdf.open("/content/drive/MyDrive/LLm training datasets /Copy of AI Engineering.pdf")

Extracting text from Pdf

In [ ]:
doc = pymupdf.open("/content/drive/MyDrive/LLm training datasets /Copy of AI Engineering.pdf")
# cretating a text output
out = open("extracted_text.txt","wb")
# iterating over pages
for page in doc:
  tp = page.get_textpage_ocr()
  text = page.get_text(textpage = tp).encode("utf8") # get plain text (is in UTF-8)
  out.write(text) # write text of pages
  out.write(f"\n--- Page {page.number + 1} ---\n".encode("utf8"))

out.close()

In [ ]:
with open("extracted_text.txt","r") as f:
    full_document_content = f.read()
# The `extracted_doc` variable (file object) is no longer needed after reading its content into `full_document_content`.

## Chunking the text

In [ ]:
import pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 512, chunk_overlap = 100)
chunks = text_splitter.split_text(full_document_content)


print(f"Total chunks: {len(chunks)}")
print(chunks[0])



Total chunks: 2894
Chip Huyen
 AI Engineering
Building Applications  
with Foundation Models
O'REILLY
7
a
,
yy
eo ye hie


## Adding Metadata

In [ ]:
from datetime import date
from langchain_core.documents import Document

# ── fill these for your document ──────────────────────────────────────
SOURCE_PATH  = "Ai engineering.pdf"
DOC_TYPE     = "book"          # e.g. "report", "invoice", "manual"
DOC_TITLE    = "Ai"
DOC_AUTHOR   = "Harsh Prajapati"
CREATION_DATE = "2005-12-20"       # ISO format from pdf.metadata["creationDate"]
TOTAL_PAGES  = 1                  # from pymupdf doc.page_count

total  = len(chunks)
today  = date.today().isoformat()

# wrap each plain string → Document with full metadata
docs = []
for idx, chunk_text in enumerate(chunks):
    meta = {
        # document-level (same for every chunk)
        "source":         SOURCE_PATH,
        "title":          DOC_TITLE,
        "author":         DOC_AUTHOR,
        "creation_date":  CREATION_DATE,
        "total_pages":    TOTAL_PAGES,
        "doc_type":       DOC_TYPE,

        # chunk-level (unique per chunk)
        "chunk_index":    idx,
        "total_chunks":   total,
        "char_count":     len(chunk_text),
        "prev_chunk_id":  idx - 1 if idx > 0          else None,
        "next_chunk_id":  idx + 1 if idx < total - 1  else None,
        "ingestion_date": today,
    }
    docs.append(Document(page_content=chunk_text, metadata=meta))

# verify
print(f"Total documents: {len(docs)}")
print(f"\nSample metadata (chunk 271):")
for k, v in docs[1].metadata.items():
    print(f"  {k:<18} {v}")
print(f"\nSample content (chunk 2:\n{docs[1].page_content}")

Total documents: 2894

Sample metadata (chunk 271):
  source             Ai engineering.pdf
  title              Ai
  author             Harsh Prajapati
  creation_date      2005-12-20
  total_pages        1
  doc_type           book
  chunk_index        1
  total_chunks       2894
  char_count         480
  prev_chunk_id      0
  next_chunk_id      2
  ingestion_date     2026-04-14
--------------------------------------------------

Sample content [Document(metadata={'source': 'Ai engineering.pdf', 'title': 'Ai', 'author': 'Harsh Prajapati', 'creation_date': '2005-12-20', 'total_pages': 1, 'doc_type': 'book', 'chunk_index': 0, 'total_chunks': 2894, 'char_count': 101, 'prev_chunk_id': None, 'next_chunk_id': 1, 'ingestion_date': '2026-04-14'}, page_content="Chip Huyen\n AI Engineering\nBuilding Applications  \nwith Foundation Models\nO'REILLY\n7\na\n,\nyy\neo ye hie"), Document(metadata={'source': 'Ai engineering.pdf', 'title': 'Ai', 'author': 'Harsh Prajapati', 'creation_date': '2005-1

our dataloader is ready !!

print 20 random chunks

In [ ]:
for doc in docs[:5]:
    print("----")
    print(doc.page_content)
    print(doc.metadata)

----
Chip Huyen
 AI Engineering
Building Applications  
with Foundation Models
O'REILLY
7
a
,
yy
eo ye hie
{'source': 'Hello transformer.pdf', 'title': 'transformer', 'author': 'Harsh Prajapati', 'creation_date': '2005-12-20', 'total_pages': 1, 'doc_type': 'book', 'chunk_index': 0, 'total_chunks': 2894, 'char_count': 101, 'prev_chunk_id': None, 'next_chunk_id': 1, 'ingestion_date': '2026-04-14'}
----
--- Page 1 ---
9
7 8 1 0 9 8 1 6 6 3 0 4
5 7 9 9 9
ISBN:   978-1-098-16630-4
US $79.99   CAN $99.99
DATA
Foundation models have enabled many new AI use cases while lowering the barriers to entry for  
building AI products. This has transformed AI from an esoteric discipline into a powerful development 
tool that anyone can use—including those with no prior AI experience.
In this accessible guide, author Chip Huyen discusses AI engineering: the process of building applications
{'source': 'Hello transformer.pdf', 'title': 'transformer', 'author': 'Harsh Prajapati', 'creation_date': '2005-12-

finding text in document using pymupdf


In [ ]:
# -*- coding: utf-8 -*-
import pymupdf

# the document to annotate
doc = pymupdf.open("/content/Hello transformers.pdf")

# the text to be marked
needle = """Hello Transformers"""

# work with first page only
page = doc[0]

# get list of text locations
# we use "quads", not rectangles because text may be tilted!
rl = page.search_for(needle, quads=True)

# mark all found quads with one annotation
page.add_squiggly_annot(rl)

# save to a new PDF
doc.save("searched_text.pdf")

FileNotFoundError: no such file: '/content/Hello transformers.pdf'

In [ ]:

import cohere
from google.colab import userdata

co = cohere.Client(userdata.get("cohere"))  # use `co`, not `cohere` directly